In [1]:
import matplotlib.pyplot as plt
plt.style.use('default')

In [2]:
import torch
import math

# **Probabilidad de colas de la normal**

In [3]:
class RestrictedExponential(torch.distributions.Distribution):

    def __init__(self, loc, scale):
        self.loc = loc
        self.scale = scale

    def sample(self, sample_shape=torch.Size()):

        u = torch.rand(sample_shape)

        x = -torch.log(u) + self.loc

        return  x

    def log_prob(self, x):

        y = torch.where(x<self.loc, torch.zeros_like(x), torch.log(torch.exp(self.loc-x)))

        return y

In [4]:
class TailNormal:

    def __init__(self):

        self.Z = torch.distributions.Normal(0.0, 1.0)

    def importance_sampling(self, x, n=1000000, decimals = 10):

        re = RestrictedExponential(x, 1)
        y = re.sample((n,))
        h = torch.where(y<x, torch.zeros_like(y), torch.ones_like(y))
        f = self.Z.log_prob(y).exp()
        g = re.log_prob(y).exp()
        mu = torch.mean(h*f/g)

        real = 1 -self.Z.cdf(x)

        error = torch.abs(real-mu)

        var = torch.var(h*f/g)/n

        if mu.numel() == 1:

            print(f"Probabilidad estimada : {mu.item():.{decimals}f}")
            print(f"Error: {error.item():.{decimals}f}")
            print(f"Varianza: {var.item():.{decimals}f}")

        return mu, error, var

    def monte_carlo(self, x, n=1000, decimals = 10):

        y = self.Z.sample((n,))
        h = torch.where(y<x, torch.zeros_like(x), torch.ones_like(x))
        mu = torch.mean(h)

        real = 1 -self.Z.cdf(x)

        error = torch.abs(real-mu)

        var =torch.var(h, unbiased=True)/n

        if mu.numel() == 1:

            print(f"Probabilidad estimada : {mu.item():.{decimals}f}")
            print(f"Error: {error.item():.{decimals}f}")
            print(f"Varianza: {var.item():.{decimals}f}")

        return mu, error, var


In [5]:
x = torch.Tensor([4.5])
tail_norm = TailNormal()

_ ,_, _ = tail_norm.monte_carlo(x, 1000000, decimals = 20)

Probabilidad estimada : 0.00000300000010611257
Error: 0.00000039746464608470
Varianza: 0.00000000000299999392


In [6]:
_, _, _ = tail_norm.importance_sampling(x, 1000000, decimals = 20)

Probabilidad estimada : 0.00000339629355039506
Error: 0.00000000117120180221
Varianza: 0.00000000000000001947


# **Función Generadora de Momentos**

In [7]:
class NormalMGF:

    def __init__(self, loc, scale):

        self.loc = loc
        self.scale = scale
        self.var = scale **2

        self.Z_f = torch.distributions.Normal(loc, scale)
        self.Z_g = None

    def importance_sampling(self, t, n=1000000, decimals = 10):

        self.Z_g = torch.distributions.Normal(self.loc + self.var*t, self.scale)

        x = self.Z_g.sample((n,))
        h = torch.exp(x*t)
        f = self.Z_f.log_prob(x).exp()
        g = self.Z_g.log_prob(x).exp()

        mu = torch.mean(h*f/g)
        var = torch.var(h*f/g)/n

        real = torch.exp(torch.Tensor([self.loc*t + self.var*(t**2)/2]))

        error = torch.abs(real-mu)

        if mu.numel() == 1:

            print(f"FGM estimada : {mu.item():.{decimals}f}")
            print(f"Error: {error.item():.{decimals}f}")
            print(f"Varianza: {var.item():.{decimals}f}")

        return mu, var, error

    def monte_carlo(self, t, n=1000000, decimals = 10):

        x = self.Z_f.sample((n,))
        h = torch.exp(x*t)

        mu = torch.mean(h)
        var = torch.var(h)/n

        real = torch.exp(torch.Tensor([self.loc*t + self.var*(t**2)/2]))

        error = torch.abs(real-mu)

        if mu.numel() == 1:

            print(f"FGM estimada : {mu.item():.{decimals}f}")
            print(f"Error: {error.item():.{decimals}f}")
            print(f"Varianza: {var.item():.{decimals}f}")

        return mu, var, error



In [8]:
norm_mfg = NormalMGF(loc=2.5, scale=1.5)
_, _, _ = norm_mfg.importance_sampling(1)

FGM estimada : 37.5247192383
Error: 0.0000038147
Varianza: 0.0000000000


In [9]:
_, _, _ = norm_mfg.monte_carlo(1)

FGM estimada : 37.5559196472
Error: 0.0311965942
Varianza: 0.0111932941


# **Precio de una opción call**

In [10]:
class CallOptionPrice:

    def __init__(self, s_0, r, sigma, k):

        self.s_0 = torch.Tensor([s_0])
        self.r = torch.Tensor([r])
        self.sigma = torch.Tensor([sigma])
        self.k = torch.Tensor([k])
        self.Z = torch.distributions.Normal(0.0, 1.0)
        self.price = None

    def compute_price(self, t):

        t = torch.Tensor([t])

        d1 = (torch.log(self.s_0/self.k) + (self.r + self.sigma**2/2)*t) / (self.sigma*torch.sqrt(t))
        d2 = d1 - self.sigma*torch.sqrt(t)
        self.price = self.s_0 * self.Z.cdf(d1) - self.k * torch.exp(-self.r*t)* self.Z.cdf(d2)


    def compute_price_mc(self, t, n=10000, decimals = 10):


        self.compute_price(t)
        t = torch.Tensor([t])

        z = self.Z.sample((n,))
        s_t = self.s_0 * torch.exp((self.r - self.sigma**2/2)*t + self.sigma*torch.sqrt(t)*z)

        payoff = torch.relu(s_t - self.k)

        price_hat = torch.exp(-self.r*t)*payoff.mean()

        var = torch.exp(-2*self.r*t)*payoff.var(unbiased=False)/n

        error = torch.abs(self.price-price_hat)

        print("----- Monte Carlo -----")
        print(f"Precio  : {self.price.item():.{decimals}f}")
        print(f"Precio estimado : {price_hat.item():.{decimals}f}")
        print(f"Error: {error.item():.{decimals}f}")
        print(f"Varianza: {var.item():.{decimals}f}")

        return price_hat, error, var

    def compute_price_is(self, t, theta=None, n=10000, decimals=10):

        t = torch.Tensor([t])

        if theta is None:

            theta = (torch.log(self.k/self.s_0) - (self.r - self.sigma**2/2)*t)/(self.sigma*torch.sqrt(t))

        else:
            theta = torch.Tensor([theta])

        d1 = (torch.log(self.s_0/self.k) + (self.r + self.sigma**2/2)*t) / (self.sigma*torch.sqrt(t))
        d2 = d1 - self.sigma*torch.sqrt(t)

        self.price = self.s_0 * self.Z.cdf(d1) - self.k * torch.exp(-self.r*t)* self.Z.cdf(d2)


        z = self.Z.sample((n,)) + theta

        s_t = self.s_0 * torch.exp((self.r - self.sigma**2/2)*t + self.sigma*torch.sqrt(t)*z)

        payoff = torch.relu(s_t - self.k)

        weights = torch.exp(-theta*z + theta**2/2)

        estimator = payoff * weights

        price_hat = torch.exp(-self.r*t) * estimator.mean()

        var = torch.exp(-2*self.r*t) * estimator.var(unbiased=False) / n

        error = torch.abs(self.price - price_hat)

        print("----- Importance Sampling -----")
        print(f"Precio  : {self.price.item():.{decimals}f}")
        print(f"Precio estimado : {price_hat.item():.{decimals}f}")
        print(f"Error : {error.item():.{decimals}f}")
        print(f"Varianza : {var.item():.{decimals}f}")

        return price_hat, error, var


In [11]:
s_0 = 100.0
r = 0.07
sigma = 0.5
k =105.0
t = 1
call_option = CallOptionPrice(s_0, r, sigma, k)
_, _, _ = call_option.compute_price_mc(t, n=10000000, decimals = 6)
_ ,_, _ = call_option.compute_price_is(t, n=10000000, decimals = 6)

----- Monte Carlo -----
Precio  : 20.600651
Precio estimado : 20.576159
Error: 0.024492
Varianza: 0.000162
----- Importance Sampling -----
Precio  : 20.600651
Precio estimado : 20.581308
Error : 0.019342
Varianza : 0.000098


In [12]:
s_0 = 100.0
r = 0.07
sigma = 0.25
k = 150.0
t = 1
call_option = CallOptionPrice(s_0, r, sigma, k)
_, _, _ = call_option.compute_price_mc(t, n=10000000, decimals = 6)
_ ,_, _ = call_option.compute_price_is(t, n=10000000, decimals = 6)

----- Monte Carlo -----
Precio  : 1.223922
Precio estimado : 1.225916
Error: 0.001994
Varianza: 0.000004
----- Importance Sampling -----
Precio  : 1.223922
Precio estimado : 1.223493
Error : 0.000429
Varianza : 0.000000


In [13]:
s_0 = 100.0
r = 0.07
sigma = 0.5
k = 500.0
t = 1
call_option = CallOptionPrice(s_0, r, sigma, k)
_, _, _ = call_option.compute_price_mc(t, n=10000000, decimals = 10)
_ ,_, _ = call_option.compute_price_is(t, n=10000000, decimals = 10)

----- Monte Carlo -----
Precio  : 0.0303061455
Precio estimado : 0.0291412547
Error: 0.0011648908
Varianza: 0.0000004321
----- Importance Sampling -----
Precio  : 0.0303061455
Precio estimado : 0.0303083435
Error : 0.0000021979
Varianza : 0.0000000002


In [14]:
s_0 = 100.0
r = 0.07
sigma = 0.5
k = 80.0
t = 1
call_option = CallOptionPrice(s_0, r, sigma, k)
_, _, _ = call_option.compute_price_mc(t, n=10000000, decimals = 6)
_ ,_, _ = call_option.compute_price_is(t, n=10000000, decimals = 6)

----- Monte Carlo -----
Precio  : 32.732899
Precio estimado : 32.735966
Error: 0.003067
Varianza: 0.000220
----- Importance Sampling -----
Precio  : 32.732899
Precio estimado : 32.773285
Error : 0.040386
Varianza : 0.000541


In [15]:
s_0 = 10
r = 0.07
sigma = 0.1
k = 15
t = 1
call_option = CallOptionPrice(s_0, r, sigma, k)
_, _, _ = call_option.compute_price_mc(t, n=10000000, decimals = 20)
_ ,_, _ = call_option.compute_price_is(t, n=10000000, decimals = 20)

----- Monte Carlo -----
Precio  : 0.00012179138138890266
Precio estimado : 0.00012633077858481556
Error: 0.00000453939719591290
Varianza: 0.00000000000913099023
----- Importance Sampling -----
Precio  : 0.00012179138138890266
Precio estimado : 0.00012197979958727956
Error : 0.00000018841819837689
Varianza : 0.00000000000000273984


# **Muestreo por importancia normalizado**

In [40]:
class NISLogisticRegression:

    def __init__(self, X, y, prior_var = 1.0):

        self.X = X
        self.y = y
        self.prior_var = prior_var
        self.n, self.d = self.X.shape

        self.prior = torch.distributions.MultivariateNormal(
            torch.zeros(self.d),
            prior_var*torch.eye(self.d))

        self.beta_hat = None
        self.samples = None
        self.weights = None


    def log_likelihood(self, beta, epsilon=1e-12):

        z = self.X @ beta.T
        p = torch.sigmoid(z)

        ll = (self.y[:, None]*torch.log(p + epsilon)
                       +(1-self.y[:, None])*torch.log(1 - p + epsilon)
                       ).sum(dim=0)
        return ll

    def log_f_kernel(self, beta, epsilon=1e-12):

        return self.log_likelihood(beta, epsilon) - self.prior.log_prob(beta)

    def importance_sampling(self, n=10000, proposal_var=1.0, epsilon=1e-12):

        proposal = torch.distributions.MultivariateNormal(
            torch.zeros(self.d),
            proposal_var*torch.eye(self.d)
        )

        samples = proposal.sample((n,))

        log_weights = self.log_f_kernel(samples, epsilon)- proposal.log_prob(samples)

        weights = torch.softmax(log_weights, dim=0)

        beta_hat = weights @ samples

        self.beta_hat = beta_hat
        self.samples = samples
        self.weights = weights

        return beta_hat

    def predict_proba(self, X_new):

        z = X_new @ self.beta_hat

        return torch.sigmoid(z)

    def predict_class(self, X_new, alpha=0.5):

        probs = self.predict_proba(X_new)

        return (probs >= alpha).int()

    def accuracy(self, X_test, y_test, alpha=0.5):

        y_pred = self.predict_class(X_test, alpha)

        return (y_pred == y_test).float().mean()









In [53]:
N = 1000
d = 3

X = torch.randn(N, d)

beta_true = torch.tensor([1.5, -2.0, 0.5])

p = torch.sigmoid(X @ beta_true)

y = torch.bernoulli(p)

X_train, X_test = X[:800,:], X[800:,:]
y_train, y_test = y[:800], y[800:]


In [54]:
model = NISLogisticRegression(X_train, y_train)

_ = model.importance_sampling(1000000)


In [55]:
model.beta_hat

tensor([ 1.6655, -2.1199,  0.4897])

In [57]:
model.accuracy(X_train,y_train)

tensor(0.8125)

In [58]:
model.accuracy(X_test, y_test)

tensor(0.8800)

In [64]:
model = NISLogisticRegression(X_train, y_train)

_ = model.importance_sampling(1000000, proposal_var=20.0)
model.accuracy(X_test, y_test)

tensor(0.8850)

In [65]:
model.accuracy(X_train, y_train)

tensor(0.8125)